# Phase 3C on the SURF A10 VM

Independent 12-case visual-state sufficiency and attention-routing pilot. Phase 3B is read-only source evidence. This notebook never re-screens or runs Part 1/Phase 1/Phase 3B. All new artifacts and caches stay on `/data/yuxuanstorage`.

The default action is `status`: Run all only displays existing records. Explicit CPU/GPU/backup opt-ins are required for work. Read `docs/phase3c_protocol.md` and `docs/phase3c_vm.md` before activating an experiment. Use the pinned `phase3b` kernel/environment, not a new unpinned Jupyter kernel.


In [ ]:
import os
import subprocess
import sys
import time
from pathlib import Path

STORAGE_ROOT = Path('/data/yuxuanstorage')
PROJECT_ROOT = STORAGE_ROOT / 'vlm-event-boundary'
PHASE3B_SOURCE = STORAGE_ROOT / 'vlm_phase3b/a10_runs/run_v4_vm_verified'
PHASE3C_PLAN = STORAGE_ROOT / 'vlm_phase3c/pilot_v3'
PHASE3C_RUN = PHASE3C_PLAN / 'execution_v1'
PHASE3C_ACTION = 'status'  # status, verify, prepare, audit, mapping, freeze, baseline, preflight, dry_run_full, full, analyze, backup
PHASE3C_CONFIRM_CPU_PREPARATION = False
PHASE3C_CONFIRM_GPU_RUN = False
PHASE3C_PREFLIGHT_REVIEWED = False
PHASE3C_CONFIRM_BACKUP = False
PHASE3C_MAPPING_MAX_PAIRS = None  # Optional cap on newly audited pairs; resume keeps completed records.
PHASE3C_PATH_MAP = None  # Optional JSON prefix map; do not rewrite historical manifests.
PHASE3C_RETRY_FAILED = False
PHASE3C_GPUS = '0,1'
PHASE3C_EXECUTION_MODE = 'model_parallel'
PHASE3C_WEIGHT_BUDGET_GIB = 10
PHASE3C_BOOTSTRAP_REPEATS = 2000
PHASE3C_NO_PLOTS = False
PHASE3C_BACKGROUND = True
PHASE3C_PAUSE_POLICY = 'off'  # off, success, finished; whole-workspace Pause needs explicit authorization.
PHASE3C_CONFIRM_EXCLUSIVE = False
PHASE3C_EMAIL_NOTIFY = False
PHASE3C_EMAIL_CONFIG = STORAGE_ROOT / '.phase3b_private/email.json'
PHASE3C_SURF_CONFIG = STORAGE_ROOT / '.phase3b_private/surf.json'
PHASE3C_BACKUP_DIR = STORAGE_ROOT / 'backups'

actions = {'status', 'verify', 'prepare', 'audit', 'mapping', 'freeze', 'baseline', 'preflight', 'dry_run_full', 'full', 'analyze', 'backup'}
if PHASE3C_ACTION not in actions:
    raise ValueError('Unknown Phase 3C action; no experiment started.')
if not STORAGE_ROOT.is_dir() or not (PROJECT_ROOT / 'scripts/inspect_phase3c.py').is_file():
    raise FileNotFoundError('Use the persistent VM volume and updated repository; do not create a fallback storage directory.')
for path in (PHASE3C_PLAN, PHASE3C_RUN, PHASE3C_BACKUP_DIR):
    if not path.resolve().is_relative_to(STORAGE_ROOT.resolve()):
        raise ValueError('New artifacts must stay on persistent STORAGE_ROOT.')
source, plan, run = PHASE3B_SOURCE.resolve(), PHASE3C_PLAN.resolve(), PHASE3C_RUN.resolve()
for path in (plan, run):
    if path == source or path.is_relative_to(source) or source.is_relative_to(path):
        raise ValueError('Phase 3B source evidence must remain separate and read-only.')
if run == plan or plan.is_relative_to(run):
    raise ValueError('Execution must not overwrite preparation artifacts.')
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from scripts.run_phase3b_vm import worker_environment
cpu_env = worker_environment('', STORAGE_ROOT)
for name in ('RESEARCH_CLOUD_TOKEN', 'SURF_API_TOKEN'):
    cpu_env.pop(name, None)
vm_common = ['--plan_dir', str(PHASE3C_PLAN), '--output_root', str(PHASE3C_RUN),
    '--storage_root', str(STORAGE_ROOT), '--project_root', str(PROJECT_ROOT), '--gpus', PHASE3C_GPUS,
    '--execution_mode', PHASE3C_EXECUTION_MODE, '--gpu_weight_budget_gib', str(PHASE3C_WEIGHT_BUDGET_GIB),
    '--bootstrap_repeats', str(PHASE3C_BOOTSTRAP_REPEATS)]
if PHASE3C_PATH_MAP is not None:
    vm_common += ['--path_map', str(PHASE3C_PATH_MAP)]
if PHASE3C_RETRY_FAILED:
    vm_common.append('--retry_failed')
if PHASE3C_NO_PLOTS:
    vm_common.append('--no_plots')
print('Phase 3C action:', PHASE3C_ACTION, '| Python:', sys.executable)
print('Read-only source:', PHASE3B_SOURCE, '| New artifacts:', PHASE3C_PLAN)


## Readiness and Checkpoints

`status` is lightweight and labels reported progress as unverified. `verify` independently checks frozen inputs and gate/capture checksums; it may read large activation files but never loads a processor/model or starts GPU work. Missing files are reported as missing, not replaced by mock evidence.


In [ ]:
inspect_command = [sys.executable, '-u', 'scripts/inspect_phase3c.py', '--plan_dir', str(PHASE3C_PLAN), '--execution_dir', str(PHASE3C_RUN)]
if PHASE3C_ACTION == 'verify':
    inspect_command.append('--verify')
subprocess.run(inspect_command, check=True, env=cpu_env)


## CPU Preparation

Explicit `prepare` runs archive audit, processor-only mapping eligibility, then freezes the 8-rescue/4-stable cohort before any new intervention. Individual `audit`, `mapping` and `freeze` actions are also available. Real source videos must be present. Mapping uses the pinned processor and CPU video decoding, not model weights. Each pair is checkpointed, and incomplete quotas block freeze.

Once frozen, preparation is closed: `prepare` reuses the cohort without rewriting audit timestamps or manifests. Do not modify case IDs, masks, layer windows or controls from intervention outcomes. Changed protocol/provenance needs a separate root.


In [ ]:
if PHASE3C_ACTION in {'prepare', 'audit', 'mapping', 'freeze'}:
    if not PHASE3C_CONFIRM_CPU_PREPARATION:
        raise RuntimeError('Set PHASE3C_CONFIRM_CPU_PREPARATION=True to run this explicit preparation action.')
    frozen = PHASE3C_PLAN / 'selection/frozen_config.json'
    if frozen.is_file():
        if PHASE3C_ACTION != 'prepare':
            raise RuntimeError('Preparation is already frozen. Use status/verify rather than rewriting it.')
        from scripts.phase3c_execution import load_selection
        load_selection(PHASE3C_PLAN)
        print('Existing frozen cohort reused. No preparation rerun or GPU work.')
    else:
        mapping_command = [sys.executable, '-u', 'scripts/audit_phase3c_mappings.py', '--plan_dir', str(PHASE3C_PLAN), '--project_root', str(PROJECT_ROOT)]
        if PHASE3C_PATH_MAP is not None:
            mapping_command += ['--path_map', str(PHASE3C_PATH_MAP)]
        if PHASE3C_MAPPING_MAX_PAIRS is not None:
            mapping_command += ['--max_pairs', str(PHASE3C_MAPPING_MAX_PAIRS)]
        if PHASE3C_RETRY_FAILED:
            mapping_command.append('--retry_failed')
        commands = {
            'audit': [sys.executable, '-u', 'scripts/prepare_phase3c.py', '--stage', 'audit', '--source_run_root', str(PHASE3B_SOURCE), '--output_dir', str(PHASE3C_PLAN)],
            'mapping': mapping_command,
            'freeze': [sys.executable, '-u', 'scripts/prepare_phase3c.py', '--stage', 'freeze', '--output_dir', str(PHASE3C_PLAN)],
        }
        started = time.perf_counter()
        for step in ('audit', 'mapping', 'freeze') if PHASE3C_ACTION == 'prepare' else (PHASE3C_ACTION,):
            print(f'Phase 3C CPU {step} starting; elapsed={(time.perf_counter()-started)/60:.1f} min; checkpoint={PHASE3C_PLAN}', flush=True)
            subprocess.run(commands[step], check=True, env=cpu_env)
        print(f'CPU action finished in {(time.perf_counter()-started)/60:.1f} min. No GPU work started.', flush=True)


## Explicit VM Execution

Run `baseline` first (24 conditions), then `preflight` (two frozen representative pairs, 152 technical forwards). Inspect their parity, identity, DeepStack timing and mask diagnostics before selecting `full`. The fixed primary grid is 696 visual replacements, 24 intact routing diagnostics and 2,592 knockout forwards, followed by CPU analysis.

`dry_run_full` checks artifact prerequisites and prints the detached command without GPU, SMTP, SURF, tmux or file writes. For real GPU stages set the explicit confirmation. `full` additionally requires `PHASE3C_PREFLIGHT_REVIEWED=True`; this flag does not bypass runner checks.

Background mode uses the verified backup/email/Pause lifecycle. Email and Pause default to disabled. Credentials stay in private files, never notebook outputs. Set `finished` only after authorizing Pause of the entire workspace. Closing your browser is safe after the detached job has started; do not manually Pause while it is running.


In [ ]:
if PHASE3C_ACTION in {'baseline', 'preflight', 'dry_run_full', 'full', 'analyze'}:
    dry_run = PHASE3C_ACTION == 'dry_run_full'
    stage = 'full' if dry_run else PHASE3C_ACTION
    if stage != 'analyze' and not dry_run and not PHASE3C_CONFIRM_GPU_RUN:
        raise RuntimeError('Set PHASE3C_CONFIRM_GPU_RUN=True for this explicit GPU stage.')
    if stage == 'full' and not dry_run and not PHASE3C_PREFLIGHT_REVIEWED:
        raise RuntimeError('Review real baseline/preflight diagnostics before confirming the full pilot.')
    if PHASE3C_BACKGROUND or dry_run:
        options = ['--stage', stage, *vm_common, '--session_name', 'phase3c_' + stage,
            '--backup_dir', str(PHASE3C_BACKUP_DIR), '--pause_policy', PHASE3C_PAUSE_POLICY,
            '--surf_config', str(PHASE3C_SURF_CONFIG)]
        if PHASE3C_EMAIL_NOTIFY:
            options += ['--email_notify', '--email_config', str(PHASE3C_EMAIL_CONFIG)]
        if PHASE3C_CONFIRM_EXCLUSIVE:
            options.append('--confirm_exclusive_workspace')
        if dry_run:
            options.append('--dry_run')
        subprocess.run([sys.executable, '-u', 'scripts/launch_phase3c_vm.py', *options], check=True)
    else:
        if PHASE3C_EMAIL_NOTIFY or PHASE3C_PAUSE_POLICY != 'off':
            raise ValueError('Email/Pause requires background lifecycle mode and verified backup.')
        subprocess.run([sys.executable, '-u', 'scripts/run_phase3c_vm.py', '--stage', stage, *vm_common], check=True)


## Status and Local Backup

The detached wrapper packages stopped work automatically. This cell normally only shows status; it does not create another backup or claim local verification. Explicit `backup` requires confirmation and refuses concurrent writers. Download every part, manifest and SHA256SUMS from the exact timestamped directory; retain the Phase 3B backup separately. Verify locally using `python3 scripts/backup_phase3c.py --verify <downloaded-directory>`.


In [ ]:
if PHASE3C_ACTION == 'backup':
    if not PHASE3C_CONFIRM_BACKUP:
        raise RuntimeError('Set PHASE3C_CONFIRM_BACKUP=True to duplicate the full stopped-run evidence.')
    subprocess.run([sys.executable, '-u', 'scripts/backup_phase3c.py', '--run_root', str(PHASE3C_RUN), '--backup_dir', str(PHASE3C_BACKUP_DIR)], check=True, env=cpu_env)
from scripts.inspect_phase3c import inspect
current = inspect(PHASE3C_PLAN, PHASE3C_RUN)
lifecycle = current['lifecycle'] or {}
print('Lifecycle (reported, not independently verified):', lifecycle.get('state', 'not started'))
if lifecycle.get('backup_verified'):
    print('VM-verified package (not locally confirmed):', lifecycle['backup_dir'])
print('Monitor log:', PHASE3C_RUN.parent / (PHASE3C_RUN.name + '_lifecycle') / 'job.log')
for name, item in current['stages'].items():
    progress = item.get('reported_progress') or {}
    if progress:
        print(name, '| passed:', progress.get('passed_tasks', '?'), '/', item.get('expected_tasks', '?'),
              '| elapsed_sec:', progress.get('elapsed_sec'), '| ETA_sec:', progress.get('eta_sec'), '| checkpoint:', progress.get('checkpoint'))
print('Keep all VM evidence until every backup part has been downloaded and verified on your computer.')
